# fishaudio/s2-pro — Local Colab run (CPU-forced)

This notebook runs **fishaudio/s2-pro locally** in Colab **without** calling any remote APIs. It forces CPU execution so it won't attempt to use a small GPU (e.g., T4) and won't stop with VRAM warnings. If you prefer GPU execution on a large GPU, edit/remove the `os.environ['CUDA_VISIBLE_DEVICES']=''` line in Cell 1.

**Important**
- This notebook *does not* call any external inference API. It downloads the model weights and runs inference locally.
- Running on CPU can be slow. If you have access to a high-memory GPU (≥24 GB) or a TPU workflow that fish-speech supports, you can modify the notebook to use that device — but this version forces CPU for reliability on Colab T4 and similar.
- Ensure you have adequate disk space (~20+ GB) to download model files.

Run cells top-to-bottom. Edit the `HF_TOKEN` and `TEXT_HINDI` in the configuration cell before running installation/downloading cell.

In [1]:
# Cell 1 — force CPU (avoid GPU VRAM checks) and show environment summary
import os, sys, subprocess, json, traceback
# Force CPU: hide GPUs from PyTorch by clearing CUDA_VISIBLE_DEVICES
os.environ['CUDA_VISIBLE_DEVICES'] = ''
print('Forced CPU mode. CUDA_VISIBLE_DEVICES set to empty.')
print('Python:', sys.version.replace('\n',' '))
in_colab = 'google.colab' in sys.modules
print('Running in Colab:', in_colab)

# Quick disk check
try:
    st = os.statvfs('/')
    free_gb = (st.f_bavail * st.f_frsize) / (1024**3)
    print('Approx free disk (GB):', round(free_gb,2))
except Exception as e:
    print('Could not determine disk space:', e)

# Basic package availability check
for pkg in ('git', 'python3'):
    try:
        out = subprocess.check_output(['which', pkg]).decode().strip()
        print(f'{pkg} ->', out)
    except Exception:
        print(f'{pkg} not found in PATH')

Forced CPU mode. CUDA_VISIBLE_DEVICES set to empty.
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Running in Colab: True
Approx free disk (GB): 206.19
git -> /usr/bin/git
python3 -> /usr/bin/python3


## Configuration — edit before running the heavy install cell

- `DO_INSTALL`: keep `True` to run installation and model download in one go.
- `MODEL_DIR`: where model files will be downloaded.
- `HF_TOKEN`: your Hugging Face token (required if the model is gated); leave empty if model is public.
- `TEXT_HINDI`: Hindi text to synthesize.
- `REFERENCE_AUDIO`: optional file path to a short reference audio (wav) to clone timbre. If empty, the model will use random timbre.
- `COMPILE`: pass `--compile` to text2semantic CLI to enable torch.compile (can speed up inference on modern PyTorch). Keep False for maximum compatibility.


In [2]:
# Cell 2 — configuration (edit as needed)
DO_INSTALL = True
MODEL_DIR = 'checkpoints/s2-pro'  # local directory for model weights
HF_TOKEN = ''  # set to 'hf_xxx...' if required (gated model)
TEXT_HINDI = 'नमस्ते। यह FishAudio S2-Pro का परीक्षण वाक्य है। कृपया इसे हिंदी में बोलिए।'  # text to synthesize
REFERENCE_AUDIO = ''  # optional: path to a short wav file to use as reference (leave empty to skip)
COMPILE = False  # set True to add --compile flag to text2semantic (experimental)

---

### Install dependencies and download model weights
This cell installs system packages and Python dependencies, then downloads the `fishaudio/s2-pro` checkpoints to `MODEL_DIR`. It avoids interactive prompts by setting environment variables. Large downloads may take a while.


In [6]:
# Cell 3 — install packages and download model (heavy)
import os, sys, subprocess, traceback, importlib, time
if not DO_INSTALL:
    print('DO_INSTALL is False — skipping installation/download.')
else:
    try:
        # 1) apt packages
        try:
            print('Installing system packages...')
            subprocess.check_call(['apt-get', 'update', '-y'])
            subprocess.check_call(['apt-get', 'install', '-y', 'ffmpeg', 'libsox-dev', 'sox', 'libsndfile1-dev', 'libasound2-dev', 'cmake', 'build-essential'])
        except Exception as e:
            print('apt-get step failed; continuing. Error:', e)

        # 2) pip installs - incremental approach
        try:
            print('Installing/Updating base Python tools...')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools==69.5.1', 'wheel'])

            print('Installing Python packages (this may take several minutes)...')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch', 'torchaudio', 'transformers', 'huggingface_hub', 'soundfile', 'numpy', 'scipy', 'gradio', 'librosa', 'pyyaml', 'loguru', 'einops', 'rotary-embedding-torch', 'vector-quantize-pytorch', 'natten', 'encodec', 'loralib'])

            print('Installing latest fish-speech from GitHub...')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-deps', 'git+https://github.com/fishaudio/fish-speech.git'])

            print('Ensuring specific dependency versions...')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'antlr4-python3-runtime==4.9.3', 'hydra-core==1.3.2', 'pytorch-lightning', 'lightning', 'accelerate>=0.26.0'])
            print('Python packages installed.')
        except Exception as e:
            print('pip install step failed. Traceback:')
            traceback.print_exc()

        # 3) Download model weights
        try:
            from huggingface_hub import snapshot_download
            os.makedirs(MODEL_DIR, exist_ok=True)
            if HF_TOKEN:
                os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
            print('Downloading fishaudio/s2-pro to', MODEL_DIR, '...')
            snapshot_download(repo_id='fishaudio/s2-pro', cache_dir=MODEL_DIR, local_dir=MODEL_DIR)
            print('Model download finished.')
        except Exception as e:
            print('Model download failed. Traceback:')
            traceback.print_exc()
    except Exception as e:
        print('Unexpected error in installation/download cell:')
        traceback.print_exc()

Installing system packages...
Installing/Updating base Python tools...
Installing Python packages (this may take several minutes)...
Installing latest fish-speech from GitHub...
Ensuring specific dependency versions...
Python packages installed.


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Model download finished.


---

## Inference — run the 3-step FishSpeech pipeline locally (CPU)
This cell executes:
1. (Optional) DAC encode of reference audio to produce prompt tokens.
2. text2semantic CLI to produce semantic codes from text.
3. DAC decode to produce final wav audio.

All steps are run locally and the resulting audio is saved in `inference_outputs/` and played in the notebook. Execution on CPU may be slow but will run end-to-end without contacting external APIs.


In [ ]:
# Cell 4 — Inference (3 steps) — Memory Optimized for CPU
import os, sys, subprocess, shlex, glob, time, traceback, gc, torch
out_dir = 'inference_outputs'
os.makedirs(out_dir, exist_ok=True)

# Helper to run shell commands
def run(cmd):
    print('\n$ ' + cmd)
    env = os.environ.copy()
    env['OMP_NUM_THREADS'] = '2'
    env['MKL_NUM_THREADS'] = '2'
    env['ACCELERATE_USE_CPU'] = 'true'
    env['HF_HUB_OFFLINE'] = '1'
    # Attempt to bypass the Colab environment check in accelerate
    env['PYTHONPATH'] = ':'.join(sys.path)

    try:
        # Using bfloat16 for text2semantic to save RAM on CPU
        if 'text2semantic' in cmd and '--half' not in cmd:
            cmd += ' --half'

        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=True, text=True, env=env)
        for line in process.stdout:
            print(line, end='')
        process.wait()
        return process.returncode == 0, ""
    except Exception as e:
        print('Command raised exception:', e)
        return False, str(e)

try:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Step 1: reference encoding (optional)
    prompt_tokens = ''
    if REFERENCE_AUDIO and os.path.exists(REFERENCE_AUDIO):
        print('Encoding reference...')
        cmd_enc = f"{sys.executable} -m fish_speech.models.dac.inference -i {shlex.quote(REFERENCE_AUDIO)} --checkpoint-path {shlex.quote(MODEL_DIR)} --output-dir {shlex.quote(out_dir)} --device cpu"
        run(cmd_enc)
        candidates = glob.glob(os.path.join(out_dir, '*.npy'))
        if candidates: prompt_tokens = sorted(candidates)[-1]

    # Step 2: text2semantic generation
    text_arg = TEXT_HINDI.replace('\n',' ')
    cmd_txt2sem = f"{sys.executable} -m fish_speech.models.text2semantic.inference --text {shlex.quote(text_arg)} --checkpoint-path {shlex.quote(MODEL_DIR)} --output-dir {shlex.quote(out_dir)} --device cpu"

    if prompt_tokens:
        cmd_txt2sem += f" --prompt-tokens {shlex.quote(prompt_tokens)} --prompt-text {shlex.quote(text_arg)}"

    ok2, _ = run(cmd_txt2sem)
    if not ok2:
        raise RuntimeError('text2semantic step failed.')

    # Step 3: decode to wav
    codes = sorted(glob.glob(os.path.join(out_dir, 'codes_*.npy')))
    if not codes: raise RuntimeError('Semantic codes not found.')
    codes_file = codes[-1]

    cmd_decode = f"{sys.executable} -m fish_speech.models.dac.inference -i {shlex.quote(codes_file)} --checkpoint-path {shlex.quote(MODEL_DIR)} --output-dir {shlex.quote(out_dir)} --device cpu"
    run(cmd_decode)

    # Result display
    wavs = sorted(glob.glob(os.path.join(out_dir, '*.wav')))
    if wavs:
        from IPython.display import Audio, display
        display(Audio(wavs[-1], autoplay=False))
    else:
        print('No wav output found.')

except Exception:
    traceback.print_exc()


$ /usr/bin/python3 -m fish_speech.models.text2semantic.inference --text 'नमस्ते। यह FishAudio S2-Pro का परीक्षण वाक्य है। कृपया इसे हिंदी में बोलिए।' --checkpoint-path checkpoints/s2-pro --output-dir inference_outputs --device cpu
2026-03-12 15:05:33.980 | INFO     | __main__:main:875 - Loading model ...
You are using a model of type fish_qwen3_omni to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
2026-03-12 15:05:34.889 | INFO     | fish_speech.models.text2semantic.llama:from_pretrained:503 - Injected Semantic IDs into Config: 151678-155773
2026-03-12 15:05:34.890 | INFO     | fish_speech.models.text2semantic.llama:from_pretrained:519 - Loading model from checkpoints/s2-pro, config: DualARModelArgs(model_type='dual_ar', vocab_size=155776, n_layer=36, n_head=32, dim=2560, intermediate_size=9728, n_local_heads=8, head_dim=128, rope_base=1000000, norm_eps=1e-06, max_seq_len=32768, dropout=0.0, tie_word_embeddings=True, attenti

***End of notebook — generated by assistant***